# Example notebook of an Ancestry QC analysis

The present notebook serves as a guide of how use the `IDEAL-GENOM-QC` library to perform an ancestry quality control. We intend to show a possible use, because each user can adapt it to its particular needs. Up to this moment the library can only detect outliers from a homogenous population that overlaps with one of the `SuperPop` present in the **1000 Genomes** data.

In this notebook the procedure to perform the ancestry quality control is more detailed so the user can get a deeper understanding of all the steps executed in this part of the pipeline.

Let us import the required libraries.

In [ ]:
import sys
import os

import pandas as pd

from pathlib import Path

# add parent directory to path
library_path = os.path.abspath('..')
if library_path not in sys.path:
    sys.path.append(library_path)

library_path = Path(library_path)

from ideal_genom.qc.ancestry_qc import AncestryQC, AncestryQCReport

In the next cell the path variables associated with the project are set.

Moreover, since each user can have a slightly different choices for the LD regions, the user can provide its own file. Nevertheless, we provide the functionality of automatically fetching high LD regions for builts **GRCh37** and **GRCH38**.

Also, the user can provide the path to the reference genome files (`reference_files` dict with `bed`/`bim`/`fam`/`psam` keys) or let the library fetch the 1000 Genomes data automatically (the default, used here).

When giving the path to the input files, the user should take into account that the input files of the ancestry check must be the output of the sample QC.

In [ ]:
DATA_PATH = library_path / 'ideal_genom' / 'data'

test_data = DATA_PATH / 'test_data'
ouputData = test_data / 'outputData'

# Use the cleaned output of the sample QC notebook as input
input_path = ouputData / 'sample_qc_results' / 'clean_files'
input_name = '1KG_GRCh38_sample_qc'
output_path= ouputData
output_name= '1KG_GRCh38_ancestry_qc'
high_ld_file = Path('path/to/ld-regions/file') # if not available, set to Path()

In the next cell we define a dictionary with the parameters to execute the ancestry QC.

The explanation of the parameters is the following:

1. `ind_pair`: parameter of **PLINK1.9** `--indep-pairwise`.
2. `pca`: number of principal components to be computed, parameter `--pca` from **PLINK1.9**.
3. `maf`: minor allele frequency, parameter `--maf` of **PLINK1.9**.
4. `ref_threshold`: distance threshold from the reference panel `SuperPop` centroid for a sample to be considered a possible outlier.
5. `stu_threshold`: distance threshold from the study population centroid for a sample to be considered a possible outlier. A sample must exceed *both* thresholds to be flagged.
6. `reference_pop`: Super population from the reference panel considered as reference for the study.
7. `num_pcs`: number of principal components used to flag a sample as outlier.
8. `distance_metric`: distance metric used for outlier detection — `'infinity'`/`'chebyshev'` (default), or a numeric Minkowski order `p >= 1` (e.g. `2` for Euclidean).

In [ ]:
ancestry_params = {
    "ind_pair"       : [50, 5, 0.2],
    "pca"            : 10,
    "maf"            : 0.01,
    "ref_threshold"  : 4,
    "stu_threshold"  : 4,
    "reference_pop"  : "SAS",
    "num_pcs"        : 10,
    "distance_metric": "infinity",
}

Initialize the class `AncestryQC`. Since no `reference_files` are provided, the 1000 Genomes reference panel will be fetched automatically for the chosen build.

In [ ]:
ancestry_qc = AncestryQC(
    input_path = input_path,
    input_name = input_name,
    output_path= output_path,
    output_name= output_name,
    high_ld_regions_file=high_ld_file,
    recompute_merge=True, # if True, it will recompute the merge of the input files
    build='38', # '38' it is the default value
)

Execute the pipeline steps of the ancestry quality control. `execute_ancestry_qc_pipeline()` merges the study data with the reference panel, cleans up the merging intermediates, runs PCA, flags ancestry outliers by distance from the reference/study centroids, and drops them, producing cleaned `PLINK` files in `ancestry_qc.clean_dir`.

The merging step alone runs PLINK many times, which prints a lot of console text; we capture it into `ancestry_qc_log` to keep the notebook readable — run `ancestry_qc_log.show()` in a new cell if you need to inspect it.

In [ ]:
%%capture ancestry_qc_log
ancestry_qc.execute_ancestry_qc_pipeline(ancestry_params=ancestry_params)

In [ ]:
print(f"Ancestry QC pipeline completed. Clean PLINK files written to: {ancestry_qc.clean_dir}")

**Note:** `execute_ancestry_qc_pipeline()` already performs the full pipeline end-to-end, including merging-intermediate cleanup, ancestry-outlier detection, and dropping the flagged samples. The cells below are for *inspecting and customizing* the resulting reports — they do not need to repeat any of that.

In [ ]:
report = AncestryQCReport(
    output_path     =ancestry_qc.plots_dir,
    einvectors      =ancestry_qc.eigenvectors,
    eigenvalues     =ancestry_qc.eigenvalues,
    ancestry_fails  =ancestry_qc.ancestry_fails,
    population_tags =ancestry_qc.population_tags,
)

In [ ]:
report.report_ancestry_qc(
    reference_pop=ancestry_params['reference_pop'],
    aspect_ratio ='equal',
    format       ='svg',
)

This generates 2D/3D PCA scatter plots (colored by `SuperPop`, with and without the flagged outliers) plus a scree plot / variance-explained table for the principal components.

In [ ]:
# Regenerate just the PCA scatter plot, e.g. with a different aspect ratio or format,
# without re-running the whole pipeline
report.draw_pca_plot(
    reference_pop   =ancestry_params['reference_pop'],
    aspect_ratio    ='auto',
    exclude_outliers=True,
    plot_dir        =ancestry_qc.plots_dir,
    plot_name       ='pca_plot_auto_aspect',
    format          ='png',
)

Let's load the population tags and the list of flagged ancestry outliers to inspect them directly.

In [ ]:
population_tags = pd.read_csv(ancestry_qc.population_tags, sep='\t')
ancestry_fails   = pd.read_csv(ancestry_qc.ancestry_fails, sep='\t', header=None, names=['ID1', 'ID2'])

In [ ]:
population_tags['SuperPop'].value_counts()

In [ ]:
print('Samples flagged as ancestry outliers:', ancestry_fails.shape[0])

The flagged outliers were already dropped by `execute_drop_ancestry_outliers()` inside the pipeline run above. The cleaned `PLINK` files are available at the path below.

Unlike sample/variant QC, there is no dedicated `AncestryQCCleanUp` class — the only intermediate cleanup performed is `_clean_merging_dir()`, which the pipeline already ran automatically after merging.

In [ ]:
clean_files = ancestry_qc.clean_dir / ancestry_qc.output_name
print(f'Clean PLINK files (ancestry outliers removed): {clean_files}')